# Bare metal x86-64 Assembly Language on *nix 64

Using the NASM assembler (https://nasm.us/)

To install on Fedora: # dnf -y install nasm

* SYSCALL instruction jumps to operating system services. To use, first put the system call number in RAX, then the arguments, if any, in RDI, RSI, RDX, R10, R8, and R9, respectively. 
* call number 60 exit a process.

## Hello world example

#### Write the source file

In [1]:
%%writefile hello.asm
; hello.asm
; source: https://cs.lmu.edu/~ray/notes/x86assembly/
            global      _start
            section     .text
_start:     mov         rax, 1          ; system call for write
            mov         rdi, 1          ; file handle 1 is stdout
            mov         rsi, message    ; address of string to output
            mov         rdx, 13         ; number of bytes
            syscall                     ; invoke operating system to do the write
            mov         rax, 60         ; system call for exit
            xor         rdi, rdi        ; exit code 0
            syscall                     ; invoke operating system to exit

            section     .data
message:    db          "Hello, world!", 10   ; note the newline (10) at the end

Overwriting hello.asm


#### Assemble and show the listing file

In [2]:
! nasm -f elf64 hello.asm -l hello.lst

In [3]:
! cat hello.lst

     1                                  ; hello.asm
     2                                  ; source: https://cs.lmu.edu/~ray/notes/x86assembly/
     3                                              global      _start
     4                                              section     .text
     5 00000000 B801000000              _start:     mov         rax, 1          ; system call for write
     6 00000005 BF01000000                          mov         rdi, 1          ; file handle 1 is stdout
     7 0000000A 48BE-                               mov         rsi, message    ; address of string to output
     7 0000000C [0000000000000000] 
     8 00000014 BA0D000000                          mov         rdx, 13         ; number of bytes
     9 00000019 0F05                                syscall                     ; invoke operating system to do the write
    10 0000001B B83C000000                          mov         rax, 60         ; system call for exit
    11 00000020 4831FF             

Columns, in sequence:
* sequential line number, only for reference
* address in memory
* bytes (data or instruction)
* label
* opcode
* operand, register, data, other information
* comments

#### Link

In [14]:
! ld --verbose -o hello.out hello.o

GNU ld version 2.35-18.fc33
  Supported emulations:
   elf_x86_64
   elf32_x86_64
   elf_i386
   elf_iamcu
   elf_l1om
   elf_k1om
   i386pep
   i386pe
   elf64bpf
using internal linker script:
/* Script for -z combreloc -z separate-code */
/* Copyright (C) 2014-2020 Free Software Foundation, Inc.
   Copying and distribution of this script, with or without modification,
   are permitted in any medium without royalty provided the copyright
   notice and this notice are preserved.  */
OUTPUT_FORMAT("elf64-x86-64", "elf64-x86-64",
	      "elf64-x86-64")
OUTPUT_ARCH(i386:x86-64)
ENTRY(_start)
SEARCH_DIR("=/usr/x86_64-redhat-linux/lib64"); SEARCH_DIR("=/usr/lib64"); SEARCH_DIR("=/usr/local/lib64"); SEARCH_DIR("=/lib64"); SEARCH_DIR("=/usr/x86_64-redhat-linux/lib"); SEARCH_DIR("=/usr/local/lib"); SEARCH_DIR("=/lib"); SEARCH_DIR("=/usr/lib");
SECTIONS
{
  PROVIDE (__executable_start = SEGMENT_START("text-segment", 0x400000)); . = SEGMENT_START("text-segment", 0x400000) + SIZEOF_HEADERS;
  .in

#### Show generated files

In [9]:
! ls

assembly.ipynb		hello.f90		  hello.lst  hello.s
fortran-assembly.ipynb	hello.f90.004t.original   hello.o    README.md
hello.asm		hello.f90.235t.optimized  hello.out


#### Run

In [15]:
! ./hello.out

Hello, world!